# 02. Tree Traversals

A traversal visits **every node once**, in a specific order. The order depends on **when you process the node** compared to its children.

**Learning order:** 4 ORDERS → RECURSIVE → ITERATIVE → ZIGZAG → RIGHT SIDE VIEW → WHEN TO USE WHICH → INTERVIEW THINKING

## 1. The 4 traversals

For this tree:

```
      1
     / \
    2   3
   / \
  4   5
```

| Order | Visit sequence | Result |
|---|---|---|
| **Preorder** (root, left, right) | root first | `[1, 2, 4, 5, 3]` |
| **Inorder** (left, root, right) | root in the middle | `[4, 2, 5, 1, 3]` |
| **Postorder** (left, right, root) | root last | `[4, 5, 2, 3, 1]` |
| **Level order** (BFS) | level by level | `[[1], [2, 3], [4, 5]]` |

Real life: **preorder** = copying a folder (create the folder first, then its contents). **Postorder** = deleting a folder or computing its size (children first). **Inorder** on a BST gives **sorted order**.

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree_from_list(vals):             # level order, None = missing child
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q, i = deque([root]), 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def level_order(root):
    if not root:
        return []
    q, res = deque([root]), []
    while q:
        res.append([])
        for _ in range(len(q)):
            node = q.popleft()
            res[-1].append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return res

def inorder(root):
    return inorder(root.left) + [root.val] + inorder(root.right) if root else []

def preorder(root):
    res = []
    def dfs(node):
        if not node: return
        res.append(node.val)
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return res

def postorder(root):
    res = []
    def dfs(node):
        if not node: return
        dfs(node.left)
        dfs(node.right)
        res.append(node.val)
    dfs(root)
    return res

t = tree_from_list([1, 2, 3, 4, 5])
print(preorder(t), inorder(t), postorder(t))   # [1, 2, 4, 5, 3] [4, 2, 5, 1, 3] [4, 5, 2, 3, 1]

[1, 2, 4, 5, 3] [4, 2, 5, 1, 3] [4, 5, 2, 3, 1]


## 2. Iterative versions (explicit stack, no recursion)

Interviewers ask for these to test whether you understand the call stack.

- **Preorder:** stack with root. Pop, visit, push **right** then **left** (so left is processed first).
- **Inorder:** go **as far left as possible** pushing nodes. Pop, visit, then move to the **right child**.
- **Postorder:** do "root, right, left" with a stack and **reverse** the result.

In [2]:
def preorder_iter(root):
    res, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        res.append(node.val)
        if node.right: stack.append(node.right)
        if node.left: stack.append(node.left)
    return res

def inorder_iter(root):
    res, stack, cur = [], [], root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        res.append(cur.val)
        cur = cur.right
    return res

def postorder_iter(root):
    res, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        res.append(node.val)
        if node.left: stack.append(node.left)
        if node.right: stack.append(node.right)
    return res[::-1]

print(preorder_iter(t), inorder_iter(t), postorder_iter(t))   # [1, 2, 4, 5, 3] [4, 2, 5, 1, 3] [4, 5, 2, 3, 1]

[1, 2, 4, 5, 3] [4, 2, 5, 1, 3] [4, 5, 2, 3, 1]


## 3. Zigzag level order

Normal BFS, but **reverse every second level**. Use a flag.

In [3]:
def zigzag(root):
    if not root:
        return []
    q, res, left_to_right = deque([root]), [], True
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        res.append(level if left_to_right else level[::-1])
        left_to_right = not left_to_right
    return res

print(zigzag(tree_from_list([3, 9, 20, None, None, 15, 7])))   # [[3], [20, 9], [15, 7]]

[[3], [20, 9], [15, 7]]


## 4. Right side view

What you see standing on the right: the **last node of every level**. (Left side view = first node.)

In [4]:
def right_side_view(root):
    if not root:
        return []
    q, res = deque([root]), []
    while q:
        size = len(q)
        for i in range(size):
            node = q.popleft()
            if i == size - 1:
                res.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return res

print(right_side_view(tree_from_list([1, 2, 3, None, 5, None, 4])))   # [1, 3, 4]

[1, 3, 4]


## 5. Which traversal when?

| Need | Use |
|---|---|
| copy / serialize a tree | preorder |
| sorted order of a BST | inorder |
| compute something from children first (size, delete, height) | postorder |
| level by level, shortest depth, views | BFS |

**Complexity:** all O(n) time. Space: recursion/stack O(h). BFS queue O(w), where `w` is the maximum width.

**Morris traversal** does inorder in O(1) space by temporarily threading pointers. Know that it exists; rarely required.

## 6. Interview thinking

**Say:** "I'll use DFS because ... / BFS because I need level information. Time O(n), space O(h)."

**Common mistakes:** pushing left before right in preorder (reverses the order), not moving to `cur.right` in iterative inorder, forgetting to reverse for iterative postorder, returning from BFS without the level size loop.

## Quick Revision

- Pre: root, L, R. In: L, root, R (sorted for BST). Post: L, R, root.
- Iterative inorder: go left pushing, pop, visit, go right.
- Iterative preorder: push right then left.
- Postorder iterative: root-right-left then reverse.
- Zigzag = BFS + flag. Right view = last of each level.